# Bitcoin Naive Forecast Audit

## Role
Final proof of the rolling persistence implementation.

## Inputs
Canonical target and frozen validated forecasts.

## Outputs
Executed row audit, manual metrics, and PASS table.

## Depends On
01 and 07.

## Authoritative Status
AUTHORITATIVE VALIDATION

## What This Notebook Does Not Do
It does not compare historical models or write forecasts.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.bitcoin_pipeline import *
RUN_GENERATION = False
PROMOTE_TO_AUTHORITATIVE = False


In [2]:
_,target=load_bitcoin_target(ROOT); train,test=canonical_split(target); v=load_validated_forecasts(ROOT); naive=target.shift(1).reindex(test.index); rows=pd.DataFrame({'Forecast date':test.index,'Previous actual':[train.iloc[-1],*test.iloc[:-1]],'Forecast':naive,'Actual':test}); display(rows.head(10)); display(rows.tail(10))

,Forecast date,Previous actual,Forecast,Actual
Timestamp,,,,
2023-08-12 00:00:00+00:00,2023-08-12 00:00:00+00:00,29398.0,29398.0,29415.0
2023-08-13 00:00:00+00:00,2023-08-13 00:00:00+00:00,29415.0,29415.0,29284.0
2023-08-14 00:00:00+00:00,2023-08-14 00:00:00+00:00,29284.0,29284.0,29408.0
2023-08-15 00:00:00+00:00,2023-08-15 00:00:00+00:00,29408.0,29408.0,29172.0
2023-08-16 00:00:00+00:00,2023-08-16 00:00:00+00:00,29172.0,29172.0,28701.0
2023-08-17 00:00:00+00:00,2023-08-17 00:00:00+00:00,28701.0,28701.0,26642.0
2023-08-18 00:00:00+00:00,2023-08-18 00:00:00+00:00,26642.0,26642.0,26051.0
2023-08-19 00:00:00+00:00,2023-08-19 00:00:00+00:00,26051.0,26051.0,26097.0
2023-08-20 00:00:00+00:00,2023-08-20 00:00:00+00:00,26097.0,26097.0,26192.0


,Forecast date,Previous actual,Forecast,Actual
Timestamp,,,,
2026-06-28 00:00:00+00:00,2026-06-28 00:00:00+00:00,59940.07,59940.07,59473.29
2026-06-29 00:00:00+00:00,2026-06-29 00:00:00+00:00,59473.29,59473.29,60163.86
2026-06-30 00:00:00+00:00,2026-06-30 00:00:00+00:00,60163.86,60163.86,58526.17
2026-07-01 00:00:00+00:00,2026-07-01 00:00:00+00:00,58526.17,58526.17,59963.46
2026-07-02 00:00:00+00:00,2026-07-02 00:00:00+00:00,59963.46,59963.46,61479.10
2026-07-03 00:00:00+00:00,2026-07-03 00:00:00+00:00,61479.10,61479.10,62522.46
2026-07-04 00:00:00+00:00,2026-07-04 00:00:00+00:00,62522.46,62522.46,63086.18
2026-07-05 00:00:00+00:00,2026-07-05 00:00:00+00:00,63086.18,63086.18,63587.06
2026-07-06 00:00:00+00:00,2026-07-06 00:00:00+00:00,63587.06,63587.06,64000.10


In [3]:
error=test-naive; manual={'MAE':np.abs(error).mean(),'RMSE':np.sqrt(np.mean(error**2)),'MAPE':100*np.mean(np.abs(error/test)),'sMAPE':100*np.mean(2*np.abs(error)/(np.abs(test)+np.abs(naive)))}; manual

{'MAE': np.float64(1290.3532422243168),
 'RMSE': np.float64(1853.6247736716243),
 'MAPE': np.float64(1.742746521465369),
 'sMAPE': np.float64(1.7441417265023809)}

In [4]:
checks={'First forecast uses final training observation':np.isclose(naive.iloc[0],train.iloc[-1]),'Later forecasts use previous revealed actual':np.allclose(naive.iloc[1:],test.iloc[:-1]),'Frozen vector identity':np.allclose(naive,v.Naive),'No same-day target use':not np.allclose(naive,test),'Exact error identity':np.allclose(error,target.diff().reindex(test.index)),'No index shift':naive.index.equals(test.index)}; result=pd.DataFrame({'Check':checks.keys(),'PASS':checks.values()}); assert result.PASS.all(); result

,Check,PASS
0,First forecast uses final training observation,True
1,Later forecasts use previous revealed actual,True
2,Frozen vector identity,True
3,No same-day target use,True
4,Exact error identity,True
5,No index shift,True
